# Recency and Trending Popularity

This notebook inspects the immutable task-03 artifact. Temporal statistics, model selection, candidate generation, fallback handling, and canonical evaluation live in `popularity.py` and `scripts/run_recency_popularity.py`.

In [ ]:
import json
from pathlib import Path

import polars as pl

from interfaces import FINAL_RECOMMENDATION_SCHEMA
from popularity import (
    ITEM_RANKING_SCHEMA,
    RecencyPopularityConfig,
    RecencyPopularityModel,
)

pl.Config.set_tbl_rows(30)

In [ ]:
artifact_dir = Path("artifacts/task03_recency_popularity_v1")
config = json.loads((artifact_dir / "config.json").read_text())
metrics = json.loads((artifact_dir / "metrics.json").read_text())
portable = json.loads((artifact_dir / "model_config.json").read_text())
metrics["selected_config_id"], portable["recency_config"]

In [ ]:
selection = pl.DataFrame(
    [
        {"config_id": config_id, **values}
        for config_id, values in metrics["selection_summary"].items()
    ]
).sort("mean_precision_at_20_all_targets", descending=True)
selection.select(
    "config_id",
    "mean_precision_at_20_all_targets",
    "mean_precision_at_20_labeled_users",
    "mean_candidate_recall",
    "mean_candidate_oracle_p20_all_targets",
).head(15)

In [ ]:
selected_id = metrics["selected_config_id"]
pl.DataFrame(
    [
        {"cutoff": fold["fold"]["cutoff"], **result}
        for fold in metrics["selection_folds"]
        for result in fold["configs"]
        if result["config_id"] == selected_id
    ]
).select(
    "cutoff",
    "precision_at_20_all_targets",
    "precision_at_20_labeled_users",
    "candidate_recall",
    "fallback_positions",
)

In [ ]:
comparison = metrics["baseline_comparison"]
pl.DataFrame(
    {
        "metric": [
            "precision_at_20_all_targets",
            "precision_at_20_labeled_users",
            "final_hits",
        ],
        "task02_global": [
            comparison["precision_at_20_all_targets"],
            comparison["precision_at_20_labeled_users"],
            comparison["final_hits"],
        ],
        "task03_recency": [
            metrics["canonical"]["precision_at_20_all_targets"],
            metrics["canonical"]["precision_at_20_labeled_users"],
            metrics["canonical"]["final_hits"],
        ],
    }
)

In [ ]:
ranking = pl.read_parquet(artifact_dir / "item_ranking.parquet")
recommendations = pl.read_parquet(artifact_dir / "recommendations.parquet")
assert ranking.schema == ITEM_RANKING_SCHEMA
assert recommendations.schema == FINAL_RECOMMENDATION_SCHEMA
model = RecencyPopularityModel.from_fitted_ranking(
    RecencyPopularityConfig.from_dict(portable["recency_config"]),
    ranking,
)
assert model.item_ranking.equals(ranking)
ranking.head(20), recommendations.head()

## Interpretation

The selected `window_positive_6h` configuration was chosen only on the three earlier rolling folds and then evaluated once on canonical. It improves canonical Precision@20 and candidate recall over task 02. The saved ranking is tied to the recorded canonical-history checksum; for a later fold or full-history inference, construct the same `RecencyPopularityModel` from `model_config.json` and refit it through `RecencyPopularityDataLoader`. The model emits only `recency_popularity` candidates; global popularity remains a separate downstream fallback.